In [1]:
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import requests
import json
import glob
from dotenv import load_dotenv

project_root = Path.cwd().parents[1]

# Load API keys from .env (EODHD_API_KEY)
# Get the EODHD API key from Tuleva 1Password and save to .env in the project root
load_dotenv(project_root / '.env')
EODHD_API_KEY = os.environ.get('EODHD_API_KEY')
assert EODHD_API_KEY, 'EODHD_API_KEY not found in .env — get it from Tuleva 1Password'

# ── Parameters ──
SEB_REPORTS_DIR = 'seb_reports'

FUNDS = [
    {'code': 'TUK75', 'name': 'Tuleva Maailma Aktsiate Pensionifond', 'isin': 'EE3600109435', 'pillar': 2},
    {'code': 'TUK00', 'name': 'Tuleva Maailma Võlakirjade Pensionifond', 'isin': 'EE3600109443', 'pillar': 2},
]

FUND_FEES = {
    'EE3600109435': 0.00205,  # TUK75: 0.205% p.a.
    'EE3600109443': 0.00163,  # TUK00: 0.163% p.a.
}

# Estonian public holidays (for working day calculations)
ESTONIAN_HOLIDAYS = [
    '2026-01-01', '2026-02-24', '2026-04-03', '2026-04-05',
    '2026-05-01', '2026-05-24', '2026-06-23', '2026-06-24',
    '2026-08-20', '2026-12-24', '2026-12-25', '2026-12-26',
]
EE_HOLIDAYS = np.array(ESTONIAN_HOLIDAYS, dtype='datetime64[D]')

def ee_busdays(date_from, date_to):
    """Count Estonian working days between two dates (exclusive start, inclusive end)."""
    d_from = np.datetime64(date_from, 'D')
    d_to = np.datetime64(date_to, 'D')
    count = int(np.busday_count(d_from, d_to + np.timedelta64(1, 'D'), holidays=EE_HOLIDAYS))
    if np.is_busday(d_from, holidays=EE_HOLIDAYS):
        count -= 1
    return max(count, 1)

def ee_offset(date_str, n=-1):
    """Return the nth Estonian working day relative to date_str."""
    d = np.datetime64(date_str, 'D')
    step = 1 if n > 0 else -1
    remaining = abs(n)
    while remaining > 0:
        d += np.timedelta64(step, 'D')
        if np.is_busday(d, holidays=EE_HOLIDAYS):
            remaining -= 1
    return str(d)

# Auto-discover the two most recent SEB position reports
csv_files = sorted(glob.glob(f'{SEB_REPORTS_DIR}/TULEVA_pos_raport_*.csv'))
assert len(csv_files) >= 1, f'No SEB position reports found in {SEB_REPORTS_DIR}/'

CSV_FILE = csv_files[-1]       # latest
CSV_FILE_PREV = csv_files[-2] if len(csv_files) >= 2 else None

# Extract NAV dates from CSV headers
def extract_nav_date(filepath):
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('As of:'):
                return line.split(';')[1].strip()
    raise ValueError(f'Could not find NAV date in {filepath}')

NAV_DATE = extract_nav_date(CSV_FILE)
NAV_DATE_PREV = extract_nav_date(CSV_FILE_PREV) if CSV_FILE_PREV else None

print(f'NAV date: {NAV_DATE} (from {Path(CSV_FILE).name})')
if NAV_DATE_PREV:
    print(f'Prev NAV date: {NAV_DATE_PREV} (from {Path(CSV_FILE_PREV).name})')
print(f'T-1: {ee_offset(NAV_DATE, -1)}, T-2: {ee_offset(NAV_DATE, -2)}')
print(f'Report run: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

NAV date: 2026-03-09 (from TULEVA_pos_raport_20260309.csv)
Prev NAV date: 2026-03-06 (from TULEVA_pos_raport_20260306.csv)
T-1: 2026-03-06, T-2: 2026-03-05
Report run: 2026-03-10 15:36


In [2]:
from IPython.display import display, HTML
import getpass

report_html = f"""
<h1 style="margin-bottom: 4px">Tuleva II pillar funds NAV calculation</h1>
<p style="color: #555; margin: 2px 0">
    <strong>NAV date:</strong> {NAV_DATE} &nbsp;|&nbsp;
    <strong>Report run:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M')} by {getpass.getuser()}
</p>
<p style="color: #333; margin-top: 12px; max-width: 720px; line-height: 1.5">
    This report takes fund position data from the SEB bank position report,
    cross-checks security prices against EODHD market data,
    pulls official NAV and management fee data from pensionikeskus.ee,
    calculates the net asset value per unit, and verifies the result
    by comparing the day-over-day NAV change to a blended benchmark index.
</p>
<hr style="margin-top: 16px">
"""
display(HTML(report_html))

In [3]:
import io, codecs
import yfinance as yf

# ═══════════════════════════════════════════════════════════════
# 1. Parse SEB position report CSVs → raw_positions
# ═══════════════════════════════════════════════════════════════

def parse_seb_csv(filepath, nav_date):
    """Parse SEB bank position report CSV into a DataFrame matching Metabase format."""
    rows = []
    with open(filepath, 'r') as f:
        lines = f.readlines()

    for line in lines[6:]:
        parts = line.strip().split(';')
        if len(parts) < 8 or not parts[0]:
            continue

        fund_code = parts[0]
        account = parts[1]
        isin = parts[2]
        name = parts[3]
        qty_str = parts[4]
        price_str = parts[5]
        value_str = parts[7]

        def parse_num(s):
            if not s or s == '':
                return None
            return float(s.replace('.', '').replace(',', '.'))

        if account == 'Total' or account == '':
            continue

        if account == 'Register':
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'UNITS', 'Account ID': isin,
                'Account Name': name, 'Quantity': parse_num(qty_str),
                'Market Price': None, 'Market Value': None,
            })
        elif account.startswith('VP'):
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'SECURITY', 'Account ID': isin,
                'Account Name': name, 'Quantity': parse_num(qty_str),
                'Market Price': parse_num(price_str), 'Market Value': parse_num(value_str),
            })
        elif 'Cash account' in name:
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'CASH', 'Account ID': '',
                'Account Name': name, 'Quantity': parse_num(value_str),
                'Market Price': 1.0, 'Market Value': parse_num(value_str),
            })
        elif 'receivable' in name.lower():
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'RECEIVABLES', 'Account ID': isin or '',
                'Account Name': name, 'Quantity': None,
                'Market Price': None, 'Market Value': parse_num(value_str),
            })
        elif 'payable' in name.lower():
            rows.append({
                'Fund Code': fund_code, 'Nav Date': nav_date,
                'Account Type': 'LIABILITY', 'Account ID': isin or '',
                'Account Name': name, 'Quantity': None,
                'Market Price': None, 'Market Value': parse_num(value_str),
            })

    return pd.DataFrame(rows)

# Parse today's and previous day's position reports
raw_positions = parse_seb_csv(CSV_FILE, NAV_DATE)
print(f'Positions (today): {len(raw_positions)} rows from {Path(CSV_FILE).name}')

if CSV_FILE_PREV:
    raw_positions_prev = parse_seb_csv(CSV_FILE_PREV, NAV_DATE_PREV)
    raw_positions = pd.concat([raw_positions, raw_positions_prev], ignore_index=True)
    print(f'Positions (prev):  {len(raw_positions_prev)} rows from {Path(CSV_FILE_PREV).name}')
print(f'Positions total:   {len(raw_positions)} rows')

# ═══════════════════════════════════════════════════════════════
# 2. Fetch ETF/fund prices from EODHD + Yahoo Finance → raw_index
#    Each row: Key, Date (date price applies to), Value, DailyChangePct
# ═══════════════════════════════════════════════════════════════

# EODHD ticker → ISIN mapping
EODHD_TICKERS = {
    # TUK75 pooled funds (EUFUND) — IE non-ETFs, priced with T-1 lag
    'IE0009FT4LX4.EUFUND': 'IE0009FT4LX4',
    'IE00BFG1TM61.EUFUND': 'IE00BFG1TM61',
    'IE00BKPTWY98.EUFUND': 'IE00BKPTWY98',
    # TUK75 exchange-traded ETFs — priced same-day
    'SGAS.XETRA': 'IE00BFNM3G45',
    'SLMC.XETRA': 'IE00BFNM3D14',
    'SGAJ.XETRA': 'IE00BFNM3L97',
    'SNAW.XETRA': 'SNAW_BENCHMARK',  # SAWD/SNAW: iShares MSCI World Screened ETF EUR (TUK75 benchmark)
    # TUK00 bond funds (EUFUND) — LU/IE pooled, priced same-day
    'LU0826455353.EUFUND': 'LU0826455353',
    'LU0839970364.EUFUND': 'LU0839970364',
    'IE0031080751.EUFUND': 'IE0031080751',
    'IE0005032192.EUFUND': 'IE0005032192',
}

# Pooled IE non-ETF ISINs: price in SEB position report is from T-1
# LU funds and TUK00 bond funds use same-day pricing
POOLED_ISINS = {
    'IE0009FT4LX4', 'IE00BFG1TM61', 'IE00BKPTWY98',
}

# Reverse map: ISIN → EODHD key
ISIN_TO_EODHD = {isin: key for key, isin in EODHD_TICKERS.items() if isin != 'SNAW_BENCHMARK'}

nav_dt = pd.to_datetime(NAV_DATE)
fetch_start = (nav_dt - pd.Timedelta(days=14)).strftime('%Y-%m-%d')
fetch_end = (nav_dt + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

index_rows = []

# ── EODHD EOD Historical API ──
print('Fetching EODHD historical prices...')
eodhd_ok = 0
for eodhd_key, isin in EODHD_TICKERS.items():
    try:
        code, exchange = eodhd_key.split('.')
        url = f'https://eodhd.com/api/eod/{code}.{exchange}?from={fetch_start}&to={fetch_end}&api_token={EODHD_API_KEY}&fmt=json'
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            for row in data:
                index_rows.append({'Key': eodhd_key, 'Date': row['date'], 'Value': float(row['adjusted_close'])})
            if data:
                eodhd_ok += 1
        else:
            print(f'  EODHD {eodhd_key}: HTTP {resp.status_code}')
    except Exception as e:
        print(f'  EODHD {eodhd_key}: {e}')

print(f'  EODHD: {eodhd_ok}/{len(EODHD_TICKERS)} tickers OK')

# Add SEB CSV security prices with correct effective dates
# Pooled IE funds: price is from T-1 (previous business day)
# ETFs and LU funds: price is from T (NAV date)
T1 = ee_offset(NAV_DATE, -1)
seb_added = 0
for _, row in raw_positions[
    (raw_positions['Account Type'] == 'SECURITY') &
    (raw_positions['Nav Date'] == NAV_DATE)
].iterrows():
    isin = row['Account ID']
    if isin in ISIN_TO_EODHD and row['Market Price'] is not None:
        price_date = T1 if isin in POOLED_ISINS else NAV_DATE
        index_rows.append({'Key': ISIN_TO_EODHD[isin], 'Date': price_date, 'Value': row['Market Price']})
        seb_added += 1
print(f'  SEB CSV: {seb_added} prices (IE pooled→T-1, others→T)')

# ── Yahoo Finance for exchange-traded ETFs (backup source) ──
YAHOO_TICKERS = {
    'SGAS.DE': 'SGAS.XETRA',
    'SLMC.DE': 'SLMC.XETRA',
    'SGAJ.DE': 'SGAJ.XETRA',
}
for yticker, key_base in YAHOO_TICKERS.items():
    try:
        hist = yf.Ticker(yticker).history(start=fetch_start, end=fetch_end)
        hist.index = hist.index.tz_localize(None)
        for dt, row in hist.iterrows():
            index_rows.append({'Key': yticker, 'Date': dt.strftime('%Y-%m-%d'), 'Value': float(row['Close'])})
    except Exception as e:
        print(f'  Yahoo {yticker}: {e}')

# ── MSCI indices for daily comparison benchmarks ──
import ssl
from urllib.request import urlopen
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

msci_start = (nav_dt - pd.Timedelta(days=14)).strftime('%Y%m%d')
msci_end = (nav_dt + pd.Timedelta(days=1)).strftime('%Y%m%d')

MSCI_INDICES = {
    'MSCI_ACWI_SCREENED': 722376,   # MSCI ACWI Screened Net EUR
    'MSCI_EM_SCREENED': 728007,      # MSCI EM Screened Net EUR
}

for key, idx_code in MSCI_INDICES.items():
    try:
        msci_url = f'https://app2.msci.com/products/service/index/indexmaster/getLevelDataForGraph?currency_symbol=EUR&index_variant=NETR&start_date={msci_start}&end_date={msci_end}&data_frequency=DAILY&index_codes={idx_code}'
        msci_raw = json.loads(urlopen(msci_url, context=ssl_ctx).read())
        n = 0
        for lvl in msci_raw['indexes']['INDEX_LEVELS']:
            dt = pd.to_datetime(str(lvl['calc_date']), format='%Y%m%d')
            index_rows.append({'Key': key, 'Date': dt.strftime('%Y-%m-%d'), 'Value': float(lvl['level_eod'])})
            n += 1
        print(f'  {key}: {n} days')
    except Exception as e:
        print(f'  {key}: {e}')

raw_index = pd.DataFrame(index_rows)
raw_index['Date'] = pd.to_datetime(raw_index['Date'])
raw_index = raw_index.drop_duplicates(subset=['Key', 'Date'], keep='first')
raw_index = raw_index.sort_values(['Key', 'Date'])

# Compute daily % change within each price series
raw_index['DailyChangePct'] = raw_index.groupby('Key')['Value'].pct_change() * 100

print(f'Index data: {len(raw_index)} total rows')

# Show latest available data point per key for transparency
for key in sorted(raw_index['Key'].unique()):
    s = raw_index[raw_index['Key'] == key].sort_values('Date')
    last = s.iloc[-1]
    chg_str = f'{last["DailyChangePct"]:+.2f}%' if pd.notna(last['DailyChangePct']) else 'n/a'
    print(f'  {key}: latest {last["Date"].strftime("%Y-%m-%d")} = {last["Value"]:.4f} (daily chg {chg_str})')

# ═══════════════════════════════════════════════════════════════
# 3. Fetch fund NAVs from pensionikeskus.ee → raw_units
# ═══════════════════════════════════════════════════════════════

def fetch_pensionikeskus(date_str, pillar=2):
    """Fetch pension fund NAV data from pensionikeskus.ee XLS download."""
    if pillar == 2:
        url = f'https://www.pensionikeskus.ee/statistika/ii-sammas/kogumispensioni-paevastatistika/?date={date_str}&download=xls'
    else:
        url = f'https://www.pensionikeskus.ee/statistika/iii-sammas/taiendava-kogumispensioni-statistika/?date={date_str}&download=xls'

    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    text = resp.content.decode('utf-16-le').lstrip('\ufeff')
    lines = text.strip().split('\n')

    rows = []
    for line in lines[1:]:
        cols = line.strip().split('\t')
        if len(cols) < 4 or not cols[0]:
            continue
        fund_name = cols[0].strip()
        date_col = cols[1].strip()
        nav_str = cols[2].strip()
        maht_str = cols[-1].strip() if len(cols) > 10 else ''

        try:
            nav_val = float(nav_str.replace(',', '.'))
            dt = datetime.strptime(date_col, '%d.%m.%Y')
            actual_date = dt.strftime('%Y-%m-%d')
            maht_val = float(maht_str.replace(',', '.')) * 1_000_000 if maht_str else None
        except (ValueError, IndexError):
            continue

        rows.append({
            'Security Name': fund_name,
            'Request Date': actual_date,
            'Nav': nav_val,
            'Balance': maht_val,
        })
    return rows

units_rows = []
dates_to_fetch = set()
for fund in FUNDS:
    dates_to_fetch.add((NAV_DATE, fund['pillar']))
    for i in range(1, 4):
        dates_to_fetch.add((ee_offset(NAV_DATE, -i), fund['pillar']))

print('Fetching pensionikeskus NAVs...')
fetched_dates = set()
for date_str, pillar in sorted(dates_to_fetch):
    cache_key = (date_str, pillar)
    if cache_key in fetched_dates:
        continue
    try:
        day_rows = fetch_pensionikeskus(date_str, pillar)
        units_rows.extend(day_rows)
        fetched_dates.add(cache_key)
    except Exception as e:
        print(f'  {date_str} (pillar {pillar}): {e}')

raw_units = pd.DataFrame(units_rows)
if len(raw_units) > 0:
    raw_units = raw_units.drop_duplicates(subset=['Security Name', 'Request Date'])
print(f'Units data: {len(raw_units)} rows from pensionikeskus.ee')

# ═══════════════════════════════════════════════════════════════
# 4. Hardcoded registry data → raw_registry
# ═══════════════════════════════════════════════════════════════

raw_registry = pd.DataFrame([
    {'Isin': isin, 'Management Fee Rate': rate}
    for isin, rate in FUND_FEES.items()
])
print(f'Registry: {len(raw_registry)} funds (hardcoded)')

# ═══════════════════════════════════════════════════════════════
# Prepare per-fund data
# ═══════════════════════════════════════════════════════════════

fund_data = {}
for fund in FUNDS:
    fd = {}
    code = fund['code']

    all_positions = raw_positions[
        (raw_positions['Fund Code'] == code) &
        (raw_positions['Nav Date'] == NAV_DATE)
    ].copy()

    fd['all_positions'] = all_positions
    fd['securities'] = all_positions[all_positions['Account Type'] == 'SECURITY'].copy()
    fd['cash'] = all_positions[all_positions['Account Type'] == 'CASH'].copy()
    fd['receivables'] = all_positions[all_positions['Account Type'] == 'RECEIVABLES'].copy()
    fd['liabilities'] = all_positions[all_positions['Account Type'] == 'LIABILITY'].copy()
    fd['units_row'] = all_positions[all_positions['Account Type'] == 'UNITS'].copy()

    print(f'\n{code} on {NAV_DATE}: {len(all_positions)} rows  '
          f'({len(fd["securities"])} sec, {len(fd["cash"])} cash, '
          f'{len(fd["receivables"])} recv, {len(fd["liabilities"])} liab, {len(fd["units_row"])} units)')

    # Units data from pensionikeskus
    all_fund_units = raw_units[raw_units['Security Name'] == fund['name']].copy()
    all_fund_units = all_fund_units.sort_values('Request Date', ascending=False)
    fd['units_today'] = all_fund_units[all_fund_units['Request Date'] == NAV_DATE]

    prev_dates = all_fund_units[all_fund_units['Request Date'] < NAV_DATE]['Request Date'].unique()
    prev_date = None
    if len(prev_dates) > 0 and len(fd['units_today']) > 0:
        today_nav_val = fd['units_today'].iloc[0]['Nav']
        for d in sorted(prev_dates, reverse=True):
            row = all_fund_units[all_fund_units['Request Date'] == d].iloc[0]
            if row['Nav'] != today_nav_val:
                prev_date = d
                break
        if prev_date is None:
            prev_date = sorted(prev_dates)[-1]
    fd['prev_date'] = prev_date
    fd['units_prev'] = all_fund_units[all_fund_units['Request Date'] == prev_date] if prev_date else pd.DataFrame()
    print(f'  Units dates: today={NAV_DATE}, prev={prev_date}')

    fund_reg = raw_registry[raw_registry['Isin'] == fund['isin']]
    if len(fund_reg) > 0:
        fd['mgmt_fee_rate'] = fund_reg.iloc[0]['Management Fee Rate']
        print(f'  Mgmt fee: {fd["mgmt_fee_rate"]*100:.3f}% p.a.')
    else:
        fd['mgmt_fee_rate'] = None
        print(f'  WARNING: Fund not found in registry!')

    fund_data[code] = fd

Positions (today): 49 rows from TULEVA_pos_raport_20260309.csv
Positions (prev):  49 rows from TULEVA_pos_raport_20260306.csv
Positions total:   98 rows
Fetching EODHD historical prices...


  EODHD: 11/11 tickers OK
  SEB CSV: 17 prices (IE pooled→T-1, others→T)


  MSCI_ACWI_SCREENED: 10 days


  MSCI_EM_SCREENED: 10 days
Index data: 167 total rows
  IE0005032192.EUFUND: latest 2026-03-09 = 23.4770 (daily chg +0.00%)
  IE0009FT4LX4.EUFUND: latest 2026-03-06 = 15.2390 (daily chg -1.19%)
  IE0031080751.EUFUND: latest 2026-03-09 = 22.7390 (daily chg +0.00%)
  IE00BFG1TM61.EUFUND: latest 2026-03-06 = 33.8700 (daily chg -1.23%)
  IE00BKPTWY98.EUFUND: latest 2026-03-06 = 13.0600 (daily chg -0.20%)
  LU0826455353.EUFUND: latest 2026-03-09 = 116.5000 (daily chg -0.15%)
  LU0839970364.EUFUND: latest 2026-03-09 = 94.3500 (daily chg -0.13%)
  MSCI_ACWI_SCREENED: latest 2026-03-09 = 4851.9471 (daily chg -1.35%)
  MSCI_EM_SCREENED: latest 2026-03-09 = 2419.5371 (daily chg -3.42%)
  SGAJ.DE: latest 2026-03-05 = 7.5400 (daily chg -1.94%)
  SGAJ.XETRA: latest 2026-03-09 = 7.3880 (daily chg -0.55%)
  SGAS.DE: latest 2026-03-05 = 12.1140 (daily chg -0.15%)
  SGAS.XETRA: latest 2026-03-09 = 11.9080 (daily chg -0.48%)
  SLMC.DE: latest 2026-03-09 = 9.9280 (daily chg -0.76%)
  SLMC.XETRA: latest 

Units data: 96 rows from pensionikeskus.ee
Registry: 2 funds (hardcoded)

TUK75 on 2026-03-09: 12 rows  (6 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Units dates: today=2026-03-09, prev=2026-03-06
  Mgmt fee: 0.205% p.a.

TUK00 on 2026-03-09: 10 rows  (4 sec, 1 cash, 2 recv, 2 liab, 1 units)
  Units dates: today=2026-03-09, prev=2026-03-06
  Mgmt fee: 0.163% p.a.


## Fund Position

All holdings of the fund on the NAV date, split into securities (ETFs) and cash/accrual lines.

In [4]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']
    cash = fd['cash']
    receivables = fd['receivables']
    liabilities = fd['liabilities']
    units_row = fd['units_row']

    sec_display = securities[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value']].copy()
    sec_display = sec_display.sort_values('Market Value', ascending=False)
    sec_total = sec_display['Market Value'].sum()

    cash_total = cash['Market Value'].sum()
    recv_total = receivables['Market Value'].sum()
    liab_total = liabilities['Market Value'].sum()
    other_rows = [
        {'Account ID': '', 'Account Name': 'Cash', 'Quantity': None, 'Market Price': None, 'Market Value': cash_total},
        {'Account ID': '', 'Account Name': 'Receivables', 'Quantity': None, 'Market Price': None, 'Market Value': recv_total},
        {'Account ID': '', 'Account Name': 'Liabilities', 'Quantity': None, 'Market Price': None, 'Market Value': liab_total},
    ]

    all_display = pd.concat([sec_display, pd.DataFrame(other_rows)], ignore_index=True)

    # Previous working day comparison
    available_dates = sorted(raw_positions[raw_positions['Fund Code'] == code]['Nav Date'].unique())
    nav_idx = available_dates.index(NAV_DATE) if NAV_DATE in available_dates else -1
    prev_nav_date = available_dates[nav_idx - 1] if nav_idx > 0 else None

    if prev_nav_date:
        prev_pos = raw_positions[
            (raw_positions['Fund Code'] == code) &
            (raw_positions['Nav Date'] == prev_nav_date)
        ]
        prev_sec = prev_pos[prev_pos['Account Type'] == 'SECURITY'].set_index('Account ID')['Market Value']
        prev_cash = prev_pos[prev_pos['Account Type'] == 'CASH']['Market Value'].sum()
        prev_recv = prev_pos[prev_pos['Account Type'] == 'RECEIVABLES']['Market Value'].sum()
        prev_liab = prev_pos[prev_pos['Account Type'] == 'LIABILITY']['Market Value'].sum()
        prev_by_type = {'Cash': prev_cash, 'Receivables': prev_recv, 'Liabilities': prev_liab}

        prev_values = []
        for _, row in all_display.iterrows():
            if row['Account ID'] and row['Account ID'] in prev_sec.index:
                prev_values.append(prev_sec[row['Account ID']])
            elif row['Account Name'] in prev_by_type:
                prev_values.append(prev_by_type[row['Account Name']])
            else:
                prev_values.append(None)

        all_display['Prev Value'] = prev_values
        all_display['Change %'] = (all_display['Market Value'] - all_display['Prev Value']) / all_display['Prev Value'].abs() * 100
    else:
        prev_pos = pd.DataFrame()
        all_display['Prev Value'] = None
        all_display['Change %'] = None

    gross_portfolio = all_display['Market Value'].abs().sum()
    all_display['% of Portfolio'] = all_display['Market Value'] / gross_portfolio * 100

    nav_components = sec_total + cash_total + recv_total + liab_total
    prev_nav_total = all_display['Prev Value'].sum() if prev_nav_date else None
    nav_change_pct = (nav_components - prev_nav_total) / abs(prev_nav_total) * 100 if prev_nav_total else None

    total_row = pd.DataFrame([{
        'Account ID': '', 'Account Name': 'Total net assets', 'Quantity': None,
        'Market Price': None, 'Market Value': nav_components,
        'Prev Value': prev_nav_total, 'Change %': nav_change_pct,
        '% of Portfolio': 100.0
    }])
    all_display = pd.concat([all_display, total_row], ignore_index=True)

    position_units = units_row.iloc[0]['Quantity'] if len(units_row) > 0 else None

    def bold_total(row):
        if row['Account Name'] == 'Total net assets':
            return ['font-weight: bold'] * len(row)
        return [''] * len(row)

    display(all_display[['Account ID', 'Account Name', 'Quantity', 'Market Price', 'Market Value', '% of Portfolio', 'Change %']].style
        .format({'Quantity': '{:,.2f}', 'Market Price': '{:.4f}', 'Market Value': '{:,.2f}',
                 '% of Portfolio': '{:.2f}%', 'Change %': '{:+.2f}%'}, na_rep='')
        .set_caption(f'{code} position as of {NAV_DATE} (vs {prev_nav_date})')
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(bold_total, axis=1)
        .hide(axis='index'))

    # Store for later cells
    fd['sec_total'] = sec_total
    fd['cash_total'] = cash_total
    fd['recv_total'] = recv_total
    fd['liab_total'] = liab_total
    fd['nav_components'] = nav_components
    fd['prev_nav_date'] = prev_nav_date
    fd['prev_nav_total'] = prev_nav_total
    fd['prev_pos'] = prev_pos if prev_nav_date else pd.DataFrame()
    fd['position_units'] = position_units

Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0,"17,953,577.58",15.2390,"273,594,568.68",29.31%,-1.19%
IE00BFG1TM61,iShares Developed World Screened Index Fund,"8,042,414.77",33.8700,"272,396,588.26",29.19%,-1.23%
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,"17,679,728.00",11.9080,"210,530,201.02",22.56%,-0.48%
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible,"7,550,798.54",13.0600,"98,613,428.93",10.57%,-0.20%
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,"7,054,146.00",9.9280,"70,033,561.49",7.50%,-0.76%
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,"1,034,931.00",7.3880,"7,646,070.23",0.82%,-0.55%
,Cash,,,"530,036.51",0.06%,+0.38%
,Receivables,,,0.00,0.00%,
,Liabilities,,,0.00,0.00%,
,Total net assets,,,"933,344,455.12",100.00%,-0.90%


Account ID,Account Name,Quantity,Market Price,Market Value,% of Portfolio,Change %
LU0839970364,iShares Global Government Bond Index Fund Class X2,"30,644.79",94.3500,"2,891,335.94",24.65%,-0.13%
IE0031080751,iShares Euro Government Bond Index Fund Flex,"125,765.36",22.7390,"2,859,778.52",24.38%,-0.35%
IE0005032192,iShares Euro Credit Bond Index Fund Flex,"121,790.70",23.4770,"2,859,280.26",24.38%,-0.26%
LU0826455353,iShares Euro Aggregate Bond Index Fund X2,"24,475.69",116.5000,"2,851,417.89",24.31%,-0.15%
,Cash,,,"267,508.77",2.28%,+0.01%
,Receivables,,,0.00,0.00%,
,Liabilities,,,0.00,0.00%,
,Total net assets,,,"11,729,321.38",100.00%,-0.22%


## Price Verification

Cross-checking fund position prices against independent index/provider data. Prices are flagged if the difference exceeds 0.5%.

In [5]:
# ── Key mapping: ISIN → index keys per source ──
KEY_MAP = {
    # TUK75 equity: pooled IE funds (T-1 lag)
    'IE0009FT4LX4': {'EODHD': 'IE0009FT4LX4.EUFUND'},
    'IE00BFG1TM61': {'EODHD': 'IE00BFG1TM61.EUFUND'},
    'IE00BKPTWY98': {'EODHD': 'IE00BKPTWY98.EUFUND'},
    # TUK75 equity: exchange-traded ETFs (same-day)
    'IE00BFNM3G45': {'EODHD': 'SGAS.XETRA', 'Yahoo': 'SGAS.DE'},
    'IE00BFNM3D14': {'EODHD': 'SLMC.XETRA', 'Yahoo': 'SLMC.DE'},
    'IE00BFNM3L97': {'EODHD': 'SGAJ.XETRA', 'Yahoo': 'SGAJ.DE'},
    # TUK00 bond funds (same-day)
    'LU0826455353': {'EODHD': 'LU0826455353.EUFUND'},
    'LU0839970364': {'EODHD': 'LU0839970364.EUFUND'},
    'IE0031080751': {'EODHD': 'IE0031080751.EUFUND'},
    'IE0005032192': {'EODHD': 'IE0005032192.EUFUND'},
}

SOURCES = ['EODHD', 'Yahoo']

# Parse index data
idx = raw_index.copy()
idx['Date'] = pd.to_datetime(idx['Date'])
nav_dt = pd.to_datetime(NAV_DATE)
t1_dt = pd.to_datetime(ee_offset(NAV_DATE, -1))

def exact_price_on_date(key, target_date):
    """Get exact price on target_date (no fallback to earlier dates)."""
    if key is None:
        return None
    rows = idx[(idx['Key'] == key) & (idx['Date'] == target_date)]
    if len(rows) == 0:
        return None
    return rows.iloc[0]['Value']

def find_consensus_price(alt_prices, threshold_pct=0.5):
    """Find consensus among ≥2 alternative prices that agree within threshold."""
    prices = [p for p in alt_prices if pd.notna(p)]
    if len(prices) < 2:
        return None
    for i in range(len(prices)):
        agreeing = [prices[i]]
        for j in range(len(prices)):
            if i != j and abs(prices[j] - prices[i]) / prices[i] * 100 <= threshold_pct:
                agreeing.append(prices[j])
        if len(agreeing) >= 2:
            return np.mean(agreeing)
    return None

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    securities = fd['securities']

    verify_rows = []
    for _, sec in securities.sort_values('Market Value', ascending=False).iterrows():
        isin = sec['Account ID']
        mapping = KEY_MAP.get(isin, {})
        fund_price = sec['Market Price']
        is_pooled = isin in POOLED_ISINS

        # Pooled IE funds use T-1 pricing; ETFs and LU funds use T (NAV date)
        eff_date = t1_dt if is_pooled else nav_dt
        lag_label = ' *' if is_pooled else ''

        row = {
            'ISIN': isin,
            'Name': sec['Account Name'] + lag_label,
            'Fund': fund_price,
            'Price Date': eff_date.strftime('%Y-%m-%d'),
        }
        for src in SOURCES:
            row[src] = exact_price_on_date(mapping.get(src), eff_date)
        verify_rows.append(row)

    verify_df = pd.DataFrame(verify_rows)

    def highlight_diff(row):
        fund_val = row['Fund']
        styles = [''] * len(row)
        for i, col in enumerate(row.index):
            if col in SOURCES and pd.notna(row[col]):
                diff_pct = abs(row[col] - fund_val) / fund_val * 100
                if diff_pct > 0.5:
                    styles[i] = 'background-color: #f8d7da'
                elif diff_pct > 0.01:
                    styles[i] = 'background-color: #fff3cd'
                else:
                    styles[i] = 'background-color: #d4edda'
        return styles

    has_lagged = any('*' in str(r.get('Name', '')) for r in verify_rows)
    footnote = '  (* = pooled IE fund, priced T-1)' if has_lagged else ''
    if len(verify_df) > 0:
        display(verify_df.style
            .format({col: '{:.4f}' for col in ['Fund'] + SOURCES}, na_rep='—')
            .set_caption(f'{code} price verification{footnote}')
            .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
            .apply(highlight_diff, axis=1)
            .hide(axis='index'))

    n_ok = n_flag = n_nodata = 0
    for _, row in verify_df.iterrows():
        for src in SOURCES:
            val = row[src]
            if pd.isna(val) or val is None:
                n_nodata += 1
            elif abs(val - row['Fund']) / row['Fund'] * 100 > 0.5:
                n_flag += 1
            else:
                n_ok += 1

    # Compute repricing adjustment where fund price disagrees with consensus
    reprice_adj = 0.0
    for _, vrow in verify_df.iterrows():
        isin = vrow['ISIN']
        fund_price = vrow['Fund']
        alt_prices = [vrow[src] for src in SOURCES if src in vrow and pd.notna(vrow[src])]
        consensus = find_consensus_price(alt_prices)
        if consensus is not None and abs(consensus - fund_price) / fund_price * 100 > 0.5:
            qty_row = securities[securities['Account ID'] == isin]
            if len(qty_row) > 0:
                reprice_adj += (consensus - fund_price) * qty_row.iloc[0]['Quantity']

    fd['verify_df'] = verify_df
    fd['n_ok'] = n_ok
    fd['n_flag'] = n_flag
    fd['n_nodata'] = n_nodata
    fd['reprice_adj'] = reprice_adj
    if abs(reprice_adj) > 0.01:
        print(f'{code}: {n_ok} OK  |  {n_flag} flagged  |  {n_nodata} no data  |  reprice adj: {reprice_adj:+,.2f} EUR')
    else:
        print(f'{code}: {n_ok} OK  |  {n_flag} flagged  |  {n_nodata} no data')

ISIN,Name,Fund,Price Date,EODHD,Yahoo
IE0009FT4LX4,CCF Developed World (ESG Screened) Index Fund Class X0 *,15.2390,2026-03-06,15.2390,—
IE00BFG1TM61,iShares Developed World Screened Index Fund *,33.8700,2026-03-06,33.8700,—
IE00BFNM3G45,iShares MSCI USA Screened UCITS ETF,11.9080,2026-03-09,11.9080,—
IE00BKPTWY98,iShares Emerging Market Screened Equity Index Fund Class Flexible *,13.0600,2026-03-06,13.0600,—
IE00BFNM3D14,iShares MSCI Europe Screened UCITS ETF,9.9280,2026-03-09,9.9280,9.9280
IE00BFNM3L97,iShares MSCI Japan Screened UCITS ETF,7.3880,2026-03-09,7.3880,—


TUK75: 7 OK  |  0 flagged  |  5 no data


ISIN,Name,Fund,Price Date,EODHD,Yahoo
LU0839970364,iShares Global Government Bond Index Fund Class X2,94.3500,2026-03-09,94.3500,—
IE0031080751,iShares Euro Government Bond Index Fund Flex,22.7390,2026-03-09,22.7390,—
IE0005032192,iShares Euro Credit Bond Index Fund Flex,23.4770,2026-03-09,23.4770,—
LU0826455353,iShares Euro Aggregate Bond Index Fund X2,116.5000,2026-03-09,116.5000,—


TUK00: 4 OK  |  0 flagged  |  4 no data


## Net Asset Value Calculation

Computing NAV per unit from position data and comparing to the reported value.

In [6]:
for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    nav_components = fd['nav_components']
    position_units = fd['position_units']
    mgmt_fee_rate = fd['mgmt_fee_rate']
    units_today = fd['units_today']
    prev_nav_date = fd['prev_nav_date']
    prev_nav_total = fd['prev_nav_total']
    prev_pos = fd['prev_pos']
    reprice_adj = fd.get('reprice_adj', 0)
    has_repricing = abs(reprice_adj) > 0.01

    nav_dt = datetime.strptime(NAV_DATE, '%Y-%m-%d')
    row_today = units_today.iloc[0]
    reported_nav = row_today['Nav']
    reported_balance = row_today['Balance']

    day_of_month = nav_dt.day
    expected_daily_fee = nav_components * mgmt_fee_rate / 365
    expected_accrual = expected_daily_fee * day_of_month
    nav_after_fees = nav_components - expected_accrual
    computed_nav = nav_after_fees / position_units
    nav_diff_eur = computed_nav - reported_nav
    nav_diff_pct = nav_diff_eur / reported_nav * 100

    # Repriced computation
    if has_repricing:
        repriced_components = nav_components + reprice_adj
        repriced_accrual = repriced_components * mgmt_fee_rate / 365 * day_of_month
        repriced_after_fees = repriced_components - repriced_accrual
        repriced_nav = repriced_after_fees / position_units
        repriced_diff_eur = repriced_nav - reported_nav
        repriced_diff_pct = repriced_diff_eur / reported_nav * 100

    # Previous day
    prev_position_units = None
    prev_accrual = None
    prev_nav_after_fees = None
    prev_computed_nav = None
    if prev_nav_date:
        prev_units_rows = prev_pos[prev_pos['Account Type'] == 'UNITS']
        prev_position_units = prev_units_rows.iloc[0]['Quantity'] if len(prev_units_rows) > 0 else None
        prev_dt = datetime.strptime(prev_nav_date, '%Y-%m-%d')
        prev_accrual = prev_nav_total * mgmt_fee_rate / 365 * prev_dt.day
        prev_nav_after_fees = prev_nav_total - prev_accrual
        if prev_position_units:
            prev_computed_nav = prev_nav_after_fees / prev_position_units

    def fmt_date(d):
        return datetime.strptime(d, '%Y-%m-%d').strftime('%d.%m.%Y') if d else 'Previous'

    def pct_change(cur, prev):
        if cur is not None and prev is not None and prev != 0:
            return f'{(cur - prev) / abs(prev) * 100:+.2f}%'
        return ''

    col_today = fmt_date(NAV_DATE)
    col_prev = fmt_date(prev_nav_date)

    reported_label = 'Total net assets as reported (EUR)' if has_repricing else 'Total net assets (EUR)'
    table_rows = [
        {'': reported_label, col_today: f'{nav_components:,.2f}', col_prev: f'{prev_nav_total:,.2f}' if prev_nav_total else '', 'Change': pct_change(nav_components, prev_nav_total)},
    ]
    if has_repricing:
        table_rows.append(
            {'': 'Total net assets repriced (EUR)', col_today: f'{repriced_components:,.2f}', col_prev: '', 'Change': f'{reprice_adj:+,.2f}'},
        )
    table_rows += [
        {'': 'Accrued mgmt fee est. (EUR)', col_today: f'{expected_accrual:,.2f}', col_prev: f'{prev_accrual:,.2f}' if prev_accrual else '', 'Change': pct_change(expected_accrual, prev_accrual)},
        {'': 'Net assets after fees (EUR)', col_today: f'{nav_after_fees:,.2f}', col_prev: f'{prev_nav_after_fees:,.2f}' if prev_nav_after_fees else '', 'Change': pct_change(nav_after_fees, prev_nav_after_fees)},
        {'': 'Units outstanding', col_today: f'{position_units:,.2f}', col_prev: f'{prev_position_units:,.2f}' if prev_position_units else '', 'Change': pct_change(position_units, prev_position_units)},
        {'': 'Computed NAV (EUR)', col_today: f'{computed_nav:.5f}', col_prev: f'{prev_computed_nav:.5f}' if prev_computed_nav else '', 'Change': pct_change(computed_nav, prev_computed_nav)},
    ]
    if has_repricing:
        table_rows.append(
            {'': 'Computed NAV repriced (EUR)', col_today: f'{repriced_nav:.5f}', col_prev: '', 'Change': f'{repriced_diff_eur:+.5f}'},
        )

    table_df = pd.DataFrame(table_rows)

    ct = col_today
    def style_table(row, col_t=ct):
        styles = [''] * len(row)
        for i, col_name in enumerate(row.index):
            if col_name == col_t:
                styles[i] = 'font-weight: bold'
            elif col_name == 'Change':
                styles[i] = 'font-style: italic'
        return styles

    fee_pct = f'{mgmt_fee_rate * 100:.3f}%' if mgmt_fee_rate else 'N/A'
    caption_html = f'{code} NAV calculation<br><i style="font-weight: normal; font-size: 0.85em">Management fee: {fee_pct} p.a.</i>'

    display(table_df.style
        .set_caption(caption_html)
        .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '1.1em'), ('text-align', 'left'), ('padding-bottom', '8px')]}])
        .apply(style_table, axis=1)
        .hide(axis='index'))

    # Store for later cells
    fd['reported_nav'] = reported_nav
    fd['reported_balance'] = reported_balance
    fd['computed_nav'] = computed_nav
    fd['prev_computed_nav'] = prev_computed_nav
    fd['nav_diff_eur'] = nav_diff_eur
    fd['nav_diff_pct'] = nav_diff_pct
    fd['expected_accrual'] = expected_accrual
    fd['total_units'] = position_units
    if has_repricing:
        fd['repriced_nav'] = repriced_nav
        fd['repriced_diff_eur'] = repriced_diff_eur
        fd['repriced_diff_pct'] = repriced_diff_pct

,09.03.2026,06.03.2026,Change
Total net assets (EUR),"933,344,455.12","941,832,070.91",-0.90%
Accrued mgmt fee est. (EUR),"47,178.64","31,738.45",+48.65%
Net assets after fees (EUR),"933,297,276.48","941,800,332.46",-0.90%
Units outstanding,"667,251,449.13","667,250,090.83",+0.00%
Computed NAV (EUR),1.39872,1.41147,-0.90%


,09.03.2026,06.03.2026,Change
Total net assets (EUR),"11,729,321.38","11,754,857.28",-0.22%
Accrued mgmt fee est. (EUR),471.42,314.97,+49.67%
Net assets after fees (EUR),"11,728,849.96","11,754,542.31",-0.22%
Units outstanding,"19,112,115.44","19,112,115.44",+0.00%
Computed NAV (EUR),0.61369,0.61503,-0.22%


## Day-over-Day Comparison

Comparing fund NAV change to a blended benchmark. Each benchmark component uses the daily change for its effective pricing date — e.g. MSCI index change for T-1 (because pooled funds are priced with 1-day lag) and ETF price change for T (same-day priced).

In [7]:
# TUK75 benchmark: blended index matching fund pricing structure
# 70% MSCI ACWI Screened daily change for T-1 (pooled funds priced with 1d lag)
# 30% SNAW ETF daily change for T (exchange-traded, same-day priced)
TUK75_BENCHMARK = {
    'components': [
        {'key': 'MSCI_ACWI_SCREENED', 'label': 'MSCI ACWI Screened', 'weight': 0.70, 'price_date': 'T1'},
        {'key': 'SNAW.XETRA',        'label': 'SNAW ETF (World Scr)', 'weight': 0.30, 'price_date': 'T'},
    ],
}

# TUK00: 100% same-day priced bond funds → daily change for T
TUK00_BENCHMARK = {
    'components': [
        {'key': 'LU0839970364.EUFUND', 'label': 'Global Govt Bond', 'weight': 0.25, 'price_date': 'T'},
        {'key': 'LU0826455353.EUFUND', 'label': 'Euro Agg Bond', 'weight': 0.25, 'price_date': 'T'},
        {'key': 'IE0031080751.EUFUND', 'label': 'Euro Govt Bond', 'weight': 0.25, 'price_date': 'T'},
        {'key': 'IE0005032192.EUFUND', 'label': 'Euro Credit Bond', 'weight': 0.25, 'price_date': 'T'},
    ],
}

idx_data = raw_index.copy()

# Estonian working day dates relative to NAV_DATE
T = pd.to_datetime(NAV_DATE)
T1 = pd.to_datetime(ee_offset(NAV_DATE, -1))
date_map = {'T': T, 'T1': T1}

def get_daily_change(key, target_date):
    """Get the daily % change for a specific date from the pre-computed series.
    If target_date has no data, use the latest available data point on or before it
    (its DailyChangePct reflects change from the previous available value).
    Returns (change_pct, actual_date, prev_date) or (None, None, None)."""
    series = idx_data[idx_data['Key'] == key].sort_values('Date')
    if len(series) == 0:
        return None, None, None

    # Try exact match first
    exact = series[series['Date'] == target_date]
    if len(exact) > 0 and pd.notna(exact.iloc[0]['DailyChangePct']):
        row = exact.iloc[0]
        # Find the previous date in the series
        idx_pos = series.index.get_loc(exact.index[0])
        prev_dt = series.iloc[idx_pos - 1]['Date'] if idx_pos > 0 else None
        return row['DailyChangePct'], row['Date'], prev_dt

    # No exact match — use latest available on or before target_date
    on_or_before = series[series['Date'] <= target_date]
    if len(on_or_before) < 2:
        return None, None, None
    latest = on_or_before.iloc[-1]
    prev = on_or_before.iloc[-2]
    chg = (latest['Value'] - prev['Value']) / prev['Value'] * 100
    return chg, latest['Date'], prev['Date']

print(f'Pricing dates: T={T.strftime("%Y-%m-%d")}, T-1={T1.strftime("%Y-%m-%d")}\n')

for fund in FUNDS:
    code = fund['code']
    fd = fund_data[code]
    prev_date = fd['prev_date']

    print(f'═══ {code} ═══')

    # Hybrid NAV: use computed NAV for today, reported for previous if no prev positions
    computed_nav = fd.get('computed_nav')
    prev_computed_nav = fd.get('prev_computed_nav')
    reported_nav = fd.get('reported_nav')

    if computed_nav is not None and prev_computed_nav is not None:
        today_nav_val = computed_nav
        prev_nav_val = prev_computed_nav
        nav_source = 'computed from positions'
    elif computed_nav is not None:
        today_nav_val = computed_nav
        units_prev = fd['units_prev']
        if len(units_prev) > 0:
            prev_nav_val = units_prev.iloc[0]['Nav']
            nav_source = 'hybrid: computed today, reported prev'
        else:
            print(f'Cannot compare: missing previous NAV data\n')
            fd['fund_nav_change_pct'] = None
            continue
    else:
        units_prev = fd['units_prev']
        units_today = fd['units_today']
        if len(units_prev) == 0 or len(units_today) == 0:
            print(f'Cannot compare: missing data\n')
            fd['fund_nav_change_pct'] = None
            continue
        prev_nav_val = units_prev.iloc[0]['Nav']
        today_nav_val = reported_nav
        nav_source = 'reported'

    fund_nav_change_pct = (today_nav_val - prev_nav_val) / prev_nav_val * 100
    n_busdays = ee_busdays(prev_date, NAV_DATE)
    busday_label = f' over {n_busdays} working day{"s" if n_busdays != 1 else ""}'

    print(f'Fund NAV ({nav_source}): {prev_nav_val:.6f} → {today_nav_val:.6f}  ({fund_nav_change_pct:+.2f}%{busday_label})')
    if n_busdays > 1:
        print(f'  Per working day: {fund_nav_change_pct / n_busdays:+.3f}%')

    # Build benchmark from components
    benchmark = TUK75_BENCHMARK if code == 'TUK75' else TUK00_BENCHMARK
    components = benchmark['components']
    comp_rows = []
    blended_pct = 0
    total_weight = 0
    notices = []

    for comp in components:
        target_date = date_map[comp['price_date']]
        chg, actual_date, prev_dt = get_daily_change(comp['key'], target_date)

        if chg is not None:
            date_label = actual_date.strftime('%m-%d')
            prev_label = prev_dt.strftime('%m-%d') if prev_dt is not None else '?'
            comp_rows.append({
                'Metric': f'{comp["label"]} ({prev_label}→{date_label})',
                'Value': f'{chg:+.2f}%',
                'Weight': f'{comp["weight"]:.0%}',
            })
            blended_pct += comp['weight'] * chg
            total_weight += comp['weight']

            # Check if data doesn't match target date
            if actual_date != target_date:
                notices.append(f'⚠ {comp["label"]}: no data for {target_date.strftime("%Y-%m-%d")}, using {prev_dt.strftime("%Y-%m-%d")}→{actual_date.strftime("%Y-%m-%d")} instead')
        else:
            comp_rows.append({
                'Metric': f'{comp["label"]} ({target_date.strftime("%Y-%m-%d")})',
                'Value': 'no data',
                'Weight': f'{comp["weight"]:.0%}',
            })

    rows = [
        {'Metric': f'Fund NAV change ({n_busdays}d)', 'Value': f'{fund_nav_change_pct:+.2f}%', 'Weight': ''},
    ]
    if n_busdays > 1:
        rows.append({'Metric': 'Fund NAV change (per day)', 'Value': f'{fund_nav_change_pct / n_busdays:+.3f}%', 'Weight': ''})
    rows.append({'Metric': '', 'Value': '', 'Weight': ''})
    rows.extend(comp_rows)

    if total_weight > 0:
        blended_pct = blended_pct / total_weight  # normalize by available weight
        blended_diff = fund_nav_change_pct - blended_pct
        rows.append({'Metric': 'Blended benchmark', 'Value': f'{blended_pct:+.2f}%', 'Weight': f'{total_weight:.0%}'})
        rows.append({'Metric': 'Tracking diff', 'Value': f'{blended_diff:+.2f}%', 'Weight': ''})
        fd['blended_idx_pct'] = blended_pct
        fd['blended_tracking_diff'] = blended_diff

    comparison = pd.DataFrame(rows)
    display(comparison.style.hide(axis='index'))

    if notices:
        from IPython.display import display as ipy_display, HTML
        ipy_display(HTML('<div style="color: #856404; background: #fff3cd; padding: 8px 12px; border-radius: 4px; margin: 4px 0 12px 0; font-size: 0.9em">' + '<br>'.join(notices) + '</div>'))

    fd['fund_nav_change_pct'] = fund_nav_change_pct
    fd['n_busdays'] = n_busdays
    print()

Pricing dates: T=2026-03-09, T-1=2026-03-06

═══ TUK75 ═══
Fund NAV (computed from positions): 1.411465 → 1.398719  (-0.90% over 1 working day)


Metric,Value,Weight
Fund NAV change (1d),-0.90%,
,,
MSCI ACWI Screened (03-04→03-05),+0.24%,70%
SNAW ETF (World Scr) (03-06→03-09),-0.50%,30%
Blended benchmark,+0.02%,100%
Tracking diff,-0.92%,



═══ TUK00 ═══
Fund NAV (computed from positions): 0.615031 → 0.613687  (-0.22% over 1 working day)


Metric,Value,Weight
Fund NAV change (1d),-0.22%,
,,
Global Govt Bond (03-06→03-09),-0.13%,25%
Euro Agg Bond (03-06→03-09),-0.15%,25%
Euro Govt Bond (03-06→03-09),+0.00%,25%
Euro Credit Bond (03-06→03-09),+0.00%,25%
Blended benchmark,-0.07%,100%
Tracking diff,-0.15%,
